In [1]:
import pandas as pd
from catboost import CatBoostClassifier

import pickle
from geoai.utils_geo.raster_ops import RasterOperations
from geoai.utils_ml.model_ops import ModelOperations

raster_ops = RasterOperations()
model_ops = ModelOperations()

In [2]:
X_train = pd.read_csv("csv_files/X_train.csv")
X_test = pd.read_csv("csv_files/X_test.csv")
y_train = pd.read_csv("csv_files/y_train_encoded.csv")
y_test = pd.read_csv("csv_files/y_test_encoded.csv")
X_train.head()

,BLUE,GREEN,RED,NIR,SWIR
0,350.0,542.0000,323.0,3277.0000,1975.3334
1,390.0,555.0000,380.0,3016.6667,1991.0000
2,1177.0,1224.6666,1268.0,1385.0000,1687.4000
3,363.2,546.5000,395.0,3244.5000,2052.0000
4,1095.6,1107.3334,1154.0,1228.0000,1573.0000


## Gradient Boosting

Gradient Boosting is a powerful machine learning technique used for regression and classification tasks. It builds an ensemble of decision trees, where each tree corrects the errors of the previous ones, resulting in a strong overall model.

### How Gradient Boosting Works

1. **Initial Model**: Start with a simple model, often just a single decision tree.
2. **Calculate Residuals**: Compute the errors (residuals) between the model's predictions and the actual values.
3. **Fit New Model to Residuals**: Train a new decision tree to predict these residuals.
4. **Update the Model**: Add the predictions of the new tree to the previous model's predictions.
5. **Iterate**: Repeat the process, each time adding a new tree that tries to correct the errors of the combined model so far.
6. **Final Prediction**: The final model is the weighted sum of all individual trees' predictions.

### Advantages of Gradient Boosting

- **High Accuracy**: Often results in very accurate and powerful models.
- **Flexibility**: Can be used for various types of data and tasks (regression, classification).

### Disadvantages of Gradient Boosting

- **Computationally Intensive**: Training can be slow, especially with a large number of trees.
- **Risk of Overfitting**: Needs careful tuning of parameters to avoid overfitting.

## CatBoost

CatBoost (Categorical Boosting) is a type of gradient boosting specifically designed to handle categorical features effectively. Developed by Yandex, it is known for its high performance and ease of use.

### How CatBoost Works

1. **Handle Categorical Features**: Automatically processes categorical features without the need for extensive preprocessing like one-hot encoding.
2. **Boosting**: Uses the principles of gradient boosting, building an ensemble of decision trees.
3. **Ordered Boosting**: Introduces ordered boosting, which helps in reducing overfitting and improving the model's stability.
4. **Efficient Training**: Optimized for fast training, even with large datasets.

### Advantages of CatBoost

- **Handles Categorical Data**: Efficiently deals with categorical features, reducing the need for manual preprocessing.
- **Robustness**: Less prone to overfitting due to its ordered boosting mechanism.
- **Ease of Use**: Requires minimal parameter tuning and preprocessing.

### Disadvantages of CatBoost

- **Complexity**: Can be more complex to understand compared to simpler models like linear regression.
- **Computational Resources**: May require significant computational power for very large datasets.

In [3]:
X_train = raster_ops.indices_binary_category(X_train) 
X_test = raster_ops.indices_binary_category(X_test)
X_train.head()

,BLUE,GREEN,RED,NIR,SWIR,NDVI,NDBI,REI,NDVI_categorized,NDVI_binary
0,350.0,542.0000,323.0,3277.0000,1975.3334,0.820556,-0.247826,0.002545,high_veg,veg
1,390.0,555.0000,380.0,3016.6667,1991.0000,0.776251,-0.204819,0.002227,high_veg,veg
2,1177.0,1224.6666,1268.0,1385.0000,1687.4000,0.044101,0.098425,0.000127,low_veg,non_veg
3,363.2,546.5000,395.0,3244.5000,2052.0000,0.782937,-0.225149,0.002438,high_veg,veg
4,1095.6,1107.3334,1154.0,1228.0000,1573.0000,0.031066,0.123170,0.000098,low_veg,non_veg


In [4]:
# create a pipeline
cat = CatBoostClassifier(
    iterations=500,               # Total number of boosting iterations
    depth=6,                      # Depth of the tree
    learning_rate=0.03,           # Learning rate
    l2_leaf_reg=3,                # L2 regularization coefficient
    min_data_in_leaf=10,          # Minimum number of samples in a leaf
    bagging_temperature=0.2,      # Controls the amount of randomness in bagging
    random_strength=1,            # Adds randomness to the score calculation
    rsm=0.8,                      # Random subspace method (feature sampling)
    leaf_estimation_method='Gradient',  # Method for leaf estimation
    early_stopping_rounds=50,     # Stop training if validation metric stops improving
    loss_function='MultiClass',   # Loss function for multiclass classification
    custom_metric=['TotalF1', 'Accuracy'],  # Additional custom metrics to evaluate
    verbose=100                   # Print training information every 100 iterations
)
pipeline = model_ops.make_selected_features_pipeline(X_train, cat)
pipeline

Pipeline(steps=[('pipeline_4',
                 FeatureUnion(transformer_list=[('pipeline_1',
                                                 ColumnTransformer(transformers=[('categorical_transformer_1',
                                                                                  Pipeline(steps=[('one_hot_transformer',
                                                                                                   OneHotEncoder(dtype=<class 'int'>,
                                                                                                                 sparse_output=False))]),
                                                                                  ['NDVI_binary']),
                                                                                 ('categorical_transformer_2',
                                                                                  Pipeline(steps=[('ordinal_transformer',
                                                                                                   OrdinalEncoder(categories=[['low_...
                                                                 ('pca_transformer',
                                                                  PCA(n_components=7))]))])),
                ('select_important_features',
                 FunctionTransformer(func=<function ModelOperations.select_important_features at 0x000002CF139AB2E0>)),
                ('print_shape',
                 FunctionTransformer(func=<function ModelOperations.print_shape at 0x000002CF13A1BF60>)),
                ('classifier',
                 <catboost.core.CatBoostClassifier object at 0x000002CF27496A50>)])

In [5]:
pipeline.fit(X_train, y_train)

y_train_pred = pipeline.predict(X_train)
y_test_pred = pipeline.predict(X_test)

# Calculate the accuracy of the model
print(f"Train Accuracy: {model_ops.calculate_classification_accuracy(y_train, y_train_pred)[3]}")
print(f"Train Accuracy: {model_ops.calculate_classification_accuracy(y_test, y_test_pred)[3]}")

d:\Projects\GEOAI\GeoAI-ISPRS-SS\geoai-env\Lib\site-packages\sklearn\utils\validation.py:1339: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


Shape before feature selection: (1864, 57)
Shape after feature selection: (1864, 21)
0:	learn: 1.3213955	total: 156ms	remaining: 1m 17s
100:	learn: 0.1744922	total: 3.03s	remaining: 12s
200:	learn: 0.1140802	total: 5.68s	remaining: 8.45s
300:	learn: 0.0811281	total: 8.22s	remaining: 5.43s
400:	learn: 0.0562910	total: 10.8s	remaining: 2.67s
499:	learn: 0.0419664	total: 13.4s	remaining: 0us
Shape before feature selection: (1864, 57)
Shape after feature selection: (1864, 21)
Shape before feature selection: (466, 57)
Shape after feature selection: (466, 21)
Train Accuracy: 0.9946316043510561
Train Accuracy: 0.9526552026123658


In [6]:
X_all = pd.concat([X_train, X_test])
y_all = pd.concat([y_train, y_test])

# train the model on the entire dataset
pipeline.fit(X_all, y_all)
with open('trained_models/cat.pkl', 'wb') as file:
    pickle.dump(pipeline, file)

d:\Projects\GEOAI\GeoAI-ISPRS-SS\geoai-env\Lib\site-packages\sklearn\utils\validation.py:1339: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


Shape before feature selection: (2330, 57)
Shape after feature selection: (2330, 21)
0:	learn: 1.3213799	total: 22.6ms	remaining: 11.3s
100:	learn: 0.1739623	total: 2.72s	remaining: 10.7s
200:	learn: 0.1184511	total: 5.39s	remaining: 8.02s
300:	learn: 0.0894925	total: 8.01s	remaining: 5.29s
400:	learn: 0.0658983	total: 10.8s	remaining: 2.67s
499:	learn: 0.0508052	total: 13.3s	remaining: 0us


END